In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Especificar opções do Sampler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido com os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Você pode usar opções para personalizar a primitiva Sampler. Esta seção foca em como especificar as opções da primitiva Qiskit Runtime. Embora a interface do método `run()` das primitivas seja comum a todas as implementações, suas opções não são. Consulte as referências de API correspondentes para obter informações sobre as opções de [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) e [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Definir opções do Sampler
Você pode definir opções ao inicializar o Sampler, após inicializá-lo, ou atualizar as opções depois que o Sampler foi inicializado. Para instruções sobre como usar essas técnicas, consulte o tópico [Introdução às opções](/guides/runtime-options-overview#options-precedence).

Além disso, você pode definir o valor `shots` no método `run()`, conforme descrito na seção a seguir.
<span id="run-method"></span>
### Método Run()
Os únicos valores que você pode passar para `run()` são os definidos na interface, ou seja, `shots`. Isso substitui qualquer valor definido para `default_shots` na execução atual.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Casos especiais
<span id="shots"></span>
#### Shots

O método `SamplerV2.run` aceita dois argumentos: uma lista de PUBs, cada um dos quais pode especificar um valor de shots específico do PUB, e um argumento de palavra-chave shots. Esses valores de shots fazem parte da interface de execução do Sampler e são independentes das opções do Sampler Runtime. Eles têm precedência sobre quaisquer valores especificados como opções para cumprir com a abstração do Sampler.

Porém, se `shots` não for especificado por nenhum PUB nem no argumento de palavra-chave run (ou se forem todos `None`), então o valor de shots das opções é usado, principalmente `default_shots`.

Em resumo, esta é a ordem de precedência para especificar shots no Sampler, para qualquer PUB particular:

1. Se o PUB especificar shots, usar esse valor.
2. Se o argumento de palavra-chave `shots` for especificado em `run`, usar esse valor.
4. Se `twirling` estiver habilitado (True por padrão), então o produto de `num_randomizations` e `shots_per_randomization`, conforme especificado nas [opções de `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), é usado.
5. Se `sampler.options.default_shots` for especificado, usar esse valor.

Portanto, se shots forem especificados em todos os lugares possíveis, o de maior precedência (shots especificados no PUB) é usado.

Exemplo: